# BARAM 2026 — 최종 v3: GBM⊕MLP 블렌드(B50) + FICR 후처리 → submission_v3.csv

`MODELING_MLP` 판정: **B50(GBM 0.5 + MLP 0.5)** 이 2023·2024 두 해 모두 GBM 단독을 이김(+0.0015~+0.0053).
이 노트북은 최종 제출 생성: 2024 holdout으로 후처리 재선택 → 전체(2022-2024) 재학습 → 2025 예측.

In [1]:
import os
os.environ.setdefault("OMP_NUM_THREADS","1")
import warnings; warnings.filterwarnings("ignore")
import json, numpy as np, pandas as pd
import torch, torch.nn as nn
torch.set_num_threads(1); torch.manual_seed(0); np.random.seed(0)
import lightgbm as lgb
from sklearn.ensemble import HistGradientBoostingRegressor as HGB
from sklearn.isotonic import IsotonicRegression
import wind_lib as W
from official_metric import group_scores, metric

GROUPS=(1,2,3); FR={}; TGT={}; FR_TE={}
for g in GROUPS:
    df,tgt=W.load_train(g); TGT[g]=tgt
    FR[g]=W.add_spatial(W.build(df,g),"train")
    FR_TE[g]=W.add_spatial(W.build(W.load_test(g),g),"test")
BASE_ALL=[c for c in W.feature_cols(FR[1]) if c not in W.SPATIAL_COLS]+["pc_pred_cf"]
V2=W.lean_features(BASE_ALL)+W.SPATIAL_COLS
BLEND=0.5
print("V2:",len(V2),"| blend GBM",1-BLEND,"/ MLP",BLEND)

def lgbm(): return lgb.LGBMRegressor(objective="mae",n_estimators=600,learning_rate=0.03,num_leaves=63,
    min_child_samples=60,subsample=0.8,subsample_freq=1,colsample_bytree=0.7,reg_lambda=1.0,
    random_state=42,n_jobs=1,verbose=-1)
def histgbm(): return HGB(loss="absolute_error",max_iter=600,learning_rate=0.03,max_leaf_nodes=63,
    l2_regularization=1.0,random_state=42)
def gbm_ens(tr,va,cols,tgt):
    lg=lgbm().fit(tr[cols],tr[tgt]); hg=histgbm().fit(tr[cols].to_numpy(),tr[tgt].to_numpy())
    return 0.5*(lg.predict(va[cols])+hg.predict(va[cols].to_numpy()))

class MLP(nn.Module):
    def __init__(s,nf,ng=3,h=128,emb=4):
        super().__init__(); s.emb=nn.Embedding(ng,emb)
        s.net=nn.Sequential(nn.Linear(nf+emb,h),nn.GELU(),nn.Dropout(0.1),
                            nn.Linear(h,h),nn.GELU(),nn.Dropout(0.1),nn.Linear(h,1))
    def forward(s,x,g): return s.net(torch.cat([x,s.emb(g)],-1)).squeeze(-1)

def train_mlp(pool_tr,feats,epochs=120,bs=512,lr=1e-3):
    pool_tr=pool_tr.sort_values("kst_dtm")
    mu=pool_tr[feats].mean(); sd=pool_tr[feats].std()+1e-8
    X=((pool_tr[feats]-mu)/sd).to_numpy(np.float32)
    y=pool_tr["cf"].to_numpy(np.float32); gid=pool_tr["gid"].to_numpy(np.int64)
    n=len(X); cut=int(n*0.85)
    Xt,yt,gt=torch.tensor(X),torch.tensor(y),torch.tensor(gid)
    net=MLP(len(feats)); opt=torch.optim.AdamW(net.parameters(),lr=lr,weight_decay=1e-4)
    best=1e9; st=None; pat=0
    for ep in range(epochs):
        net.train(); perm=np.random.permutation(np.arange(cut))
        for i in range(0,len(perm),bs):
            b=perm[i:i+bs]; opt.zero_grad()
            ((net(Xt[b],gt[b])-yt[b]).abs().mean()).backward(); opt.step()
        net.eval()
        with torch.no_grad(): vl=(net(Xt[cut:],gt[cut:])-yt[cut:]).abs().mean().item()
        if vl<best-1e-5: best=vl; st={k:v.clone() for k,v in net.state_dict().items()}; pat=0
        else: pat+=1
        if pat>=10: break
    net.load_state_dict(st); return net,(mu,sd)

def mlp_predict(net,scaler,fr,feats,g,cap):
    mu,sd=scaler
    X=torch.tensor(((fr[feats]-mu)/sd).to_numpy(np.float32))
    gid=torch.full((len(fr),),g-1,dtype=torch.long)
    net.eval()
    with torch.no_grad(): p=net(X,gid).numpy()
    return np.clip(p,0,1)*cap

def make_pool(parts):
    return pd.concat(parts,ignore_index=True)

def blend_predict(tr_frames, va_frames):
    """tr/va_frames: {g: (tr2,va2)} with pc. 반환 {g: blend_pred}."""
    pool=[]
    for g,(tr2,_) in tr_frames.items():
        p=tr2[V2+["kst_dtm"]].copy(); p["cf"]=tr2[TGT[g]]/W.CAP[g]; p["gid"]=g-1; pool.append(p)
    net,scaler=train_mlp(make_pool(pool),V2)
    out={}
    for g,(tr2,va2) in tr_frames.items():
        cap=W.CAP[g]
        gp=np.clip(gbm_ens(tr2,va2,V2,TGT[g]),0,cap)
        mp=mlp_predict(net,scaler,va2,V2,g,cap)
        out[g]=np.clip((1-BLEND)*gp+BLEND*mp,0,cap)
    return out

V2: 65 | blend GBM 0.5 / MLP 0.5


## 1. 2024 holdout: B50 예측 + OOF → FICR 후처리 재선택

In [2]:
# holdout 프레임
tr_frames={};
for g in GROUPS:
    tgt=TGT[g]; cap=W.CAP[g]; fr=FR[g]
    m_tr=fr.kst_dtm<W.VALID_START; m_va=fr.kst_dtm>=W.VALID_START
    iso=W.fit_powercurve(fr[m_tr],tgt,cap)
    tr_frames[g]=(W.with_pc(fr[m_tr],iso), W.with_pc(fr[m_va],iso))
val_pred=blend_predict(tr_frames,{g:v for g,(t,v) in tr_frames.items()})

# OOF (연도 swap; g3는 시간순 70/30)
OOF={}
for g in GROUPS:
    tgt=TGT[g]; cap=W.CAP[g]; tr2=tr_frames[g][0]
    oof=np.full(len(tr2),np.nan); years=sorted(tr2.kst_dtm.dt.year.unique())
    if len(years)>=2:
        for ty in years:
            m_in=tr2.kst_dtm.dt.year!=ty; m_out=(tr2.kst_dtm.dt.year==ty).to_numpy()
            sub_tr={g:(tr2[m_in],tr2[tr2.kst_dtm.dt.year==ty])}
            oof[m_out]=blend_predict(sub_tr,None)[g]
    else:
        n=len(tr2); cut=int(n*0.7)
        sub_tr={g:(tr2.iloc[:cut],tr2.iloc[cut:])}
        oof[cut:]=blend_predict(sub_tr,None)[g]
    OOF[g]=oof
    print(f"group{g} OOF 유효 {int(np.isfinite(oof).sum())}/{len(tr2)}")

group1 OOF 유효 17421/17421


group2 OOF 유효 17422/17422


group3 OOF 유효 2628/8759


In [3]:
def debias_fit(tr,tgt,oof):
    d=tr.assign(oof=oof); d=d[np.isfinite(d["oof"])].copy(); d["resid"]=d[tgt]-d["oof"]
    d["lb"]=pd.cut(d["lead_h"],bins=[15,21,27,33,40],labels=False,include_lowest=True)
    d["wq"]=pd.qcut(d["gfs_wind_speed_100m_mean"],5,labels=False,duplicates="drop")
    return d.groupby(["lb","wq"])["resid"].mean(), d["resid"].mean()
def debias_apply(va,pred,tbl,glob):
    v=va.copy()
    v["lb"]=pd.cut(v["lead_h"],bins=[15,21,27,33,40],labels=False,include_lowest=True)
    v["wq"]=pd.qcut(v["gfs_wind_speed_100m_mean"],5,labels=False,duplicates="drop")
    corr=np.array([tbl.get(k,glob) for k in zip(v["lb"],v["wq"])])
    return pred+corr
def nudge_fit(tr,tgt,oof,cap):
    d=tr.assign(oof=oof); d=d[np.isfinite(d["oof"])]
    yt=d[tgt].to_numpy(); yp=d["oof"].to_numpy()
    best=(1.0,0.0); bf=-1
    for s in np.linspace(0.90,1.15,26):
        for sh in np.linspace(-0.06,0.06,25)*cap:
            f=group_scores(yt,np.clip(yp*s+sh,0,cap),cap)[1]
            if f>bf: bf=f; best=(s,sh)
    return best

STORE={}; choice={}
for g in GROUPS:
    tgt=TGT[g]; cap=W.CAP[g]; tr2,va2=tr_frames[g]; bp=val_pred[g]
    tbl,glob=debias_fit(tr2,tgt,OOF[g]); s,sh=nudge_fit(tr2,tgt,OOF[g],cap)
    p1=np.clip(debias_apply(va2,bp,tbl,glob),0,cap)
    cand={"P0_none":bp,"P1_debias":p1,"P3_nudge":np.clip(bp*s+sh,0,cap),
          "P5_deb_nudge":np.clip(p1*s+sh,0,cap)}
    STORE[g]=dict(tbl=tbl,glob=glob,nudge=(s,sh))
    rows=[]
    for name,p in cand.items():
        nm,fi=group_scores(va2[tgt].to_numpy(),p,cap); rows.append(dict(post=name,nmae=nm,ficr=fi,contrib=fi-nm))
    t=pd.DataFrame(rows).set_index("post"); choice[g]=t["contrib"].idxmax()
    print(f"group{g}: 선택={choice[g]}"); print(t.round(4).to_string())

def apply_post(g,frame,pred):
    cap=W.CAP[g]; st=STORE[g]; ch=choice[g]
    if ch=="P0_none": return pred
    if ch=="P1_debias": return np.clip(debias_apply(frame,pred,st["tbl"],st["glob"]),0,cap)
    if ch=="P3_nudge": return np.clip(pred*st["nudge"][0]+st["nudge"][1],0,cap)
    q=np.clip(debias_apply(frame,pred,st["tbl"],st["glob"]),0,cap)
    return np.clip(q*st["nudge"][0]+st["nudge"][1],0,cap)

ans=pd.DataFrame({f"kpx_group_{g}":tr_frames[g][1][TGT[g]].to_numpy() for g in GROUPS})
p0=pd.DataFrame({f"kpx_group_{g}":val_pred[g] for g in GROUPS})
p1=pd.DataFrame({f"kpx_group_{g}":apply_post(g,tr_frames[g][1],val_pred[g]) for g in GROUPS})
t0=metric(ans,p0); t1=metric(ans,p1)
print(f"\n2024 공식 총점: B50기준 {t0[0]:.4f} (1-NMAE {t0[1]:.4f}, FICR {t0[2]:.4f})")
print(f"                +후처리 {t1[0]:.4f} (1-NMAE {t1[1]:.4f}, FICR {t1[2]:.4f})  Δ{t1[0]-t0[0]:+.4f}")

group1: 선택=P5_deb_nudge
                nmae    ficr  contrib
post                                 
P0_none       0.1244  0.3218   0.1974
P1_debias     0.1215  0.3321   0.2105
P3_nudge      0.1193  0.4297   0.3104
P5_deb_nudge  0.1211  0.4329   0.3118


group2: 선택=P5_deb_nudge
                nmae    ficr  contrib
post                                 
P0_none       0.1244  0.4187   0.2943
P1_debias     0.1232  0.4180   0.2948
P3_nudge      0.1224  0.4433   0.3209
P5_deb_nudge  0.1218  0.4454   0.3237
group3: 선택=P3_nudge
                nmae    ficr  contrib
post                                 
P0_none       0.1425  0.2627   0.1201
P1_debias     0.1363  0.2955   0.1593
P3_nudge      0.1410  0.3352   0.1942
P5_deb_nudge  0.1720  0.3476   0.1756



2024 공식 총점: B50기준 0.6020 (1-NMAE 0.8695, FICR 0.3344)
                +후처리 0.6383 (1-NMAE 0.8720, FICR 0.4045)  Δ+0.0363


## 2. 최종 학습(2022-2024 전체) → 2025 예측 → submission_v3.csv

In [4]:
full_frames={}
for g in GROUPS:
    tgt=TGT[g]; cap=W.CAP[g]
    iso=W.fit_powercurve(FR[g],tgt,cap)
    full_frames[g]=(W.with_pc(FR[g],iso), W.with_pc(FR_TE[g],iso))
test_pred=blend_predict(full_frames,None)

out=W.load_test(1)[["forecast_id","kst_dtm"]].rename(columns={"kst_dtm":"forecast_kst_dtm"})
for g in GROUPS:
    pred=apply_post(g, full_frames[g][1], test_pred[g])
    out[f"kpx_group_{g}"]=pred
assert out.shape[0]==8760
for g in GROUPS:
    c=out[f"kpx_group_{g}"]; assert (c>=0).all() and (c<=W.CAP[g]).all() and c.notna().all()
out.to_csv("submission/ver_3/submission.csv",index=False); print("saved submission/ver_3/submission.csv",out.shape)

summary=dict(pipeline="V2 65 feats · GBM(LGBM+HistGBM)⊕MLP B50 · FICR postproc",
    chosen_post=choice,
    holdout_total_base=round(float(t0[0]),4), holdout_total_post=round(float(t1[0]),4),
    one_minus_nmae=round(float(t1[1]),4), ficr=round(float(t1[2]),4))
json.dump(summary,open("submission/ver_3/result.json","w"),ensure_ascii=False,indent=2)
print(json.dumps(summary,ensure_ascii=False,indent=2))

saved submission_v3.csv (8760, 5)
{
  "pipeline": "V2 65 feats · GBM(LGBM+HistGBM)⊕MLP B50 · FICR postproc",
  "chosen_post": {
    "1": "P5_deb_nudge",
    "2": "P5_deb_nudge",
    "3": "P3_nudge"
  },
  "holdout_total_base": 0.602,
  "holdout_total_post": 0.6383,
  "one_minus_nmae": 0.872,
  "ficr": 0.4045
}
